In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.6 MB/s eta 0:00:00


In [ ]:
!unzip -q /content/fiw_embeddings.zip -d /tmp/
!unzip -q /content/FIDs.zip -d /tmp/

In [ ]:
import os
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import HeteroData, Data
from torch_geometric.nn import HeteroConv, SAGEConv, GATConv

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv
from sklearn.metrics import f1_score, classification_report

In [ ]:
ignore_values = {0, -1, 7, 8}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# TODO fix this
# root = Path("/content/drive/MyDrive/ml_with_graphs_project/data")
# familyIDsDir = root / "FIDs"
# embeddingDir = root / "fiw_embeddings"
familyIDsDir = Path("/tmp/FIDs")
embeddingDir = Path("/tmp/fiw_embeddings")

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
import shutil
# local_data_dir = Path("/tmp/fiw_data")

# if not local_data_dir.exists():
#     print("Copying data to local storage...")
#     shutil.copytree(
#         "/content/drive/MyDrive/ml_with_graphs_project/data",
#         local_data_dir
#     )
#     print("Done.")

# embeddingDir  = local_data_dir / "fiw_embeddings"
# familyIDsDir  = local_data_dir / "fiw_families"

local_data_dir = Path("/tmp/fiw_data")
local_embedding_dir = local_data_dir / "fiw_embeddings"
local_families_dir  = local_data_dir / "fiw_families"

if not local_embedding_dir.exists():
    family_ids = sorted([p.name for p in Path("/content/drive/MyDrive/ml_with_graphs_project/data/fiw_embeddings").iterdir()])

    for id in tqdm(family_ids, desc="Copying embeddings"):
        for mid_dir in (embeddingDir / id).iterdir():
            src  = mid_dir / "mean_embedding.pkl"
            dst  = local_embedding_dir / id / mid_dir.name / "mean_embedding.pkl"
            dst.parent.mkdir(parents=True, exist_ok=True)
            if src.exists():
                shutil.copy2(src, dst)

if not local_families_dir.exists():
    print("Copying family CSVs...")
    shutil.copytree(familyIDsDir, local_families_dir)
    print("Done.")

# Update paths
embeddingDir = local_embedding_dir
familyIDsDir = local_families_dir


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ml_with_graphs_project/data/fiw_embeddings'

In [ ]:
def load_embedding(path):
  with open(path, "rb") as f:
    emb = pickle.load(f)

  emb = np.asarray(emb)

  if emb.ndim > 1:
    emb = emb.squeeze()

  return emb

def valid_relationship(value):
  if pd.isna(value):
    return False

  try:
    value = int(value)
  except Exception:
    return False

  return value not in ignore_values

def calc_database_relationship(graph_relation, directed):
  if directed:
    return graph_relation + 1
  else:
    # note that 4 would mean EITHER 4 or 1. We can't know exactly which, but we are combining to form a father-child relationship.
    return graph_relation + 2

def calc_graph_relationship(database_relation, directed):
  if directed:
    return database_relation - 1
  else:
    if database_relation == 1:
      database_relation = 4
    elif database_relation == 6:
      database_relation = 3
    elif database_relation == 7 or database_relation == 8:
      database_relation = 6
    return database_relation - 2 # make it start at 0

In [ ]:
from collections import Counter

size_counts = Counter(f['x'].size(0) for f in families)
total_members = sum(size_counts.values())
total_families = sum(size * count for size, count in size_counts.items())
for size, count in sorted(size_counts.items()):
    print(f"  {size} members: {count} families")
print(total_members)
print(total_families)

  2 members: 51 families
  3 members: 150 families
  4 members: 258 families
  5 members: 197 families
  6 members: 135 families
  7 members: 79 families
  8 members: 42 families
  9 members: 25 families
  10 members: 18 families
  11 members: 8 families
  12 members: 5 families
  13 members: 2 families
  14 members: 3 families
  15 members: 2 families
  16 members: 1 families
  18 members: 1 families
  19 members: 1 families
  20 members: 1 families
  38 members: 1 families
980
5030


In [ ]:
def find_asymmetric_pairs_from_files():
    asymmetric = set()
    all_relations = set()
    family_ids = sorted([p.name for p in familyIDsDir.iterdir()])

    for id in tqdm(family_ids, desc="Scanning families"):
        mid_csv = familyIDsDir / id / "mid.csv"
        if not mid_csv.exists():
            continue

        try:
            df = pd.read_csv(mid_csv)
            mids = df["MID"].astype(int).tolist()
            row_map = {int(row["MID"]): row for _, row in df.iterrows()}

            for mid_a in mids:
                for mid_b in mids:
                    if mid_a == mid_b:
                        continue
                    if mid_a not in row_map or mid_b not in row_map:
                        continue

                    val_ab = row_map[mid_a].get(str(mid_b))
                    val_ba = row_map[mid_b].get(str(mid_a))

                    if not valid_relationship(val_ab) or not valid_relationship(val_ba):
                        continue

                    rel_ab = int(val_ab)
                    rel_ba = int(val_ba)

                    all_relations.add(rel_ab)
                    all_relations.add(rel_ba)

                    if rel_ab != rel_ba:
                        pair = tuple(sorted([rel_ab, rel_ba]))
                        asymmetric.add(pair)

        except Exception as e:
            print(f"\nFailed on {id}: {e}")

    print("Asymmetric relationship pairs (directed):")
    for a, b in sorted(asymmetric):
        print(f"  {a} <-> {b}")

    print(f"\nAll relation types found: {sorted(all_relations)}")
    print(f"Num relations (raw):      {len(all_relations)}")
    print(f"Num relations (0-indexed):{max(all_relations) + 1}")

    return asymmetric, all_relations

asymmetric_pairs = find_asymmetric_pairs_from_files()
print("Asymmetric relationship pairs (directed):")
for a, b in sorted(asymmetric_pairs):
    print(f"  {a} <-> {b}")


NameError: name 'tqdm' is not defined

# Family Building

In [ ]:
def load_family(id, directed):
  mid_csv = familyIDsDir / id / "mid.csv"
  df = pd.read_csv(mid_csv)

  mids = df["MID"].astype(int).tolist()

  valid_mids = []

  for mid in mids:
    embed_path = embeddingDir / id / f"MID{mid}" / "mean_embedding.pkl"

    if os.path.exists(embed_path):
      valid_mids.append(mid)

  # mid_to_idx = {mid: i for i, mid in enumerate(mids)}
  mid_to_idx = {mid: i for i, mid in enumerate(valid_mids)}

  embeddings = []

  # for mid in mids:
  for mid in valid_mids:
    embed_path = embeddingDir / id / f"MID{mid}" / "mean_embedding.pkl"

    embeddings.append(load_embedding(embed_path))

  x = torch.tensor(np.stack(embeddings), dtype = torch.float)

  edges = []

  # rel_columns = [c for c in df.columns if c not in ["MID", "Name", "Gender"]]

  for _, row in df.iterrows():
    source = int(row["MID"])
    for col in df.columns:
      if col in ["MID", "Name", "Gender"]:
        continue

      fam_member = int(col)

      if source == fam_member:
        continue

      if not valid_relationship(row[col]):
        continue

      fam_rel_id = calc_graph_relationship(int(row[col]), directed)

      if source not in mid_to_idx or fam_member not in mid_to_idx:
        continue

      person = mid_to_idx[source]
      target = mid_to_idx[fam_member]

      edges.append((person, target, fam_rel_id))

  return {
      "family_id": id,
      "x": x,
      "mids": mids,
      "edges": edges,
  }

In [ ]:
from tqdm import tqdm
def load_all_families(directed):
  # family_ids = sorted([p.name for p in familyIDsDir.iterdir()])
  family_ids = sorted([p.name for p in embeddingDir.iterdir()])

  families = []

  for id in tqdm(family_ids, desc = "Loading Families"):
    try:
      mid_csv = familyIDsDir / id / "mid.csv"

      if not mid_csv.exists():
        continue

      family = load_family(id, directed)

      if family['x'].size(0) >= 2 and len(family["edges"]) > 0:
        families.append(family)
    except Exception as e:
      print(f"Error loading family {id}: {e}")

  return families

# Relationship Mapping and Masking

In [ ]:
# as far as I can tell, there are only 0 through 5. 0 is self, so we don't have it.
# instead, I combine 4 and 1 if its undirected (so I replace 1 with 4, and subtract 2 (for 0 and 1) to get the new graph representation)
# if its directed, 1 is just subtracted from the dataset relationship. Planning to test both approaches and compare
# build label mapping is kinda useless I realised, since I can just do the above. BUT, I need to make sure if there are any relationships greater than 5.
# if so, then I might need this since it will work for any number of relationships.

# def build_label_mapping(families):
#   rel_ids = ({rel for family in families for _, _, rel in family["edges"]})
#   rel_to_label = {rel: i for i, rel in enumerate(rel_ids)}
#   label_to_rel = {i, rel for rel, i in rel_to_label.items()}

In [ ]:
def make_masked_sample(family, masked_node, directed):
  x = family['x']

  num_nodes = x.size(0)

  visible_src, visible_dst, visible_rel = [], [], []
  pred_src, pred_dst, pred_rel = [], [], []

  for source, target, relationship in family['edges']:
    if target == masked_node:
      continue

    if source == masked_node:
      pred_src.append(source)
      pred_dst.append(target)
      pred_rel.append(relationship)

    else:
      visible_src.append(source)
      visible_dst.append(target)
      visible_rel.append(relationship)

  if len(pred_rel) == 0:
    return None

  data = Data()
  data.x = x

  data.num_nodes = num_nodes

  if len(visible_src) > 0:
    data.edge_index = torch.tensor([visible_src, visible_dst], dtype = torch.long)
    data.edge_attr = torch.tensor(visible_rel, dtype = torch.long)

  else:
    data.edge_index = torch.empty((2, 0), dtype = torch.long)
    data.edge_attr = torch.empty((0, ), dtype = torch.long)

  data.edge_label_index = torch.tensor([pred_src, pred_dst], dtype = torch.long)
  data.edge_label = torch.tensor(pred_rel, dtype = torch.long)

  data.mask_node = masked_node

  return data

In [ ]:
# # mask_node is the index that we're hiding
# def hetero_make_masked_sample(family, mask_node, rel_to_label, all_rel_ids, directed):
#   # x = family['x']
#   # edges = family['edges']

#   data = HeteroData
#   data['person'].x = family['x']

#   edge_dict = {id: [[], []] for id in all_rel_ids}

#   visible_edges = {}

#   predictions = {
#       "source": [],
#       "target": [],
#       "edge_type": [],
#   }

#   for source, target, relationship in family['edges']:
#     # graph_rel = calc_graph_relationship(relationship, directed)
#     graph_rel = relationship

#     if graph_rel is None:
#       continue

#     if target == mask_node:
#       continue

#     if source == mask_node:
#       predictions['source'].append(source)
#       predictions['target'].append(target)
#       predictions['edge_type'].append(graph_rel)

#       continue

#     visible_edges.setdefault(graph_rel, [[], []])

#     visible_edges[graph_rel][0].append(source)
#     visible_edges[graph_rel][1].append(target)

#   if len(predictions["edge_type"]) == 0:
#     return None

#   all_graph_labels = sorted(set(visible_edges.keys()) | set(predictions["edge_type"]))

#   for relationship in all_graph_labels:
#     edge_type = ("person", f"rel_{relationship}", "person")
#     srcs, tgts = visible_edges.get(relationship, ([], []))

#     if len(srcs) == 0:
#       edge_index = torch.empty((2, 0), dtype = torch.long)

#     else:
#       edge_index = torch.tensor([srcs, tgts], dtype = torch.long)

#     data[edge_type].edge_index = edge_index

#   data["edge_label_index"] = torch.tensor([predictions['source'], predictions['target']], dtype = torch.long)

#   data["edge_label"] = torch.tensor(predictions['edge_type'], dtype = torch.long)

#   return data

# DATASET

In [ ]:
class FamilyDataset:
  def __init__(self, families, directed, num_relations) -> None:
    self.num_relations = num_relations
    self.directed = directed
    self.samples = []

    for family in families:
      for node in range(family['x'].size(0)):
        sample = make_masked_sample(family, node, directed)

        if sample is not None:
          self.samples.append(sample)

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    return self.samples[idx]

# MODEL

In [ ]:
class FamilyRelationGNN(torch.nn.Module):
  def __init__(self, in_dim, hidden_dim, num_relationships, dropout = 0.3):
    super().__init__()

    self.num_relations = num_relationships
    edge_dim = num_relationships

    self.input_proj = nn.Linear(in_dim, hidden_dim)

    # self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=4, concat=False, edge_dim = edge_dim, dropout = dropout)
    # self.conv2 = GATv2Conv(hidden_dim, hidden_dim, heads=4, concat=False, edge_dim = edge_dim, dropout = dropout)
    self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=2, concat=False, edge_dim = edge_dim, dropout = dropout)
    self.conv2 = GATv2Conv(hidden_dim, hidden_dim, heads=2, concat=False, edge_dim = edge_dim, dropout = dropout)
    self.dropout = nn.Dropout(dropout)

    self.classifier = nn.Sequential(
        nn.Linear(hidden_dim * 3, hidden_dim),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden_dim, num_relationships)
    )

  def encode(self, x, edge_index, edge_attr):
    x = self.input_proj(x).relu()

    if edge_attr.size(0) > 0:
      edge_attr_oh = F.one_hot(edge_attr, num_classes = self.num_relations).float()
    else:
      edge_attr_oh = None
    x = self.dropout(x)
    x = self.conv1(x, edge_index, edge_attr = edge_attr_oh).relu()
    # x = self.dropout(x)
    # x = self.conv2(x, edge_index, edge_attr = edge_attr_oh).relu()

    return x

  def forward(self, data):
    x = self.encode(data.x, data.edge_index, data.edge_attr)

    src = data.edge_label_index[0]
    tgt = data.edge_label_index[1]

    x_src = self.input_proj(data.x[src]).relu()
    x_tgt = x[tgt]

    pair = torch.cat([x_src, x_tgt, x_src * x_tgt], dim = -1)

    return self.classifier(pair)

    # x = self.conv1(data.x, data.edge_index).relu()
    # x = self.conv2(x, data.edge_index)

    # src = data.edge_label_index[0]
    # dst = data.edge_label_index[1]

    # x_src = x[src]
    # x_dst = x[dst]

    # pair = torch.cat([x_src, x_dst, x_src * x_dst], dim = 1)

    # return self.classifier(pair)

In [ ]:
# class FamilyRelationGNN(torch.nn.Module):
#   def __init__(self, in_dim, hidden_dim, num_relationships, dropout = 0.3):
#     super().__init__()

#     self.num_relations = num_relationships
#     edge_dim = num_relationships

#     self.input_proj = nn.Linear(in_dim, hidden_dim)

#     # self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=4, concat=False, edge_dim = edge_dim, dropout = dropout)
#     # self.conv2 = GATv2Conv(hidden_dim, hidden_dim, heads=4, concat=False, edge_dim = edge_dim, dropout = dropout)
#     self.conv1 = GATv2Conv(in_dim, hidden_dim, heads=2, concat=False, edge_dim = edge_dim, dropout = dropout)
#     self.conv2 = GATv2Conv(hidden_dim, hidden_dim, heads=2, concat=False, edge_dim = edge_dim, dropout = dropout)
#     self.dropout = nn.Dropout(dropout)

#     self.classifier = nn.Sequential(
#         nn.Linear(hidden_dim * 3, hidden_dim),
#         nn.ReLU(),
#         nn.Dropout(dropout),
#         nn.Linear(hidden_dim, num_relationships)
#     )

#   def encode(self, x, edge_index, edge_attr):
#     # x = self.input_proj(x).relu()

#     if edge_attr.size(0) > 0:
#       edge_attr_oh = F.one_hot(edge_attr, num_classes = self.num_relations).float()
#     else:
#       edge_attr_oh = None
#     # x = self.dropout(x)
#     x = self.conv1(x, edge_index, edge_attr = edge_attr_oh).relu()
#     # x = self.dropout(x)
#     # x = self.conv2(x, edge_index, edge_attr = edge_attr_oh).relu()

#     return x

#   def forward(self, data):
#     x = self.encode(data.x, data.edge_index, data.edge_attr)

#     src = data.edge_label_index[0]
#     tgt = data.edge_label_index[1]

#     x_src = self.input_proj(data.x[src]).relu()
#     x_tgt = x[tgt]

#     pair = torch.cat([x_src, x_tgt, x_src * x_tgt], dim = -1)

#     return self.classifier(pair)

#     # x = self.conv1(data.x, data.edge_index).relu()
#     # x = self.conv2(x, data.edge_index)

#     # src = data.edge_label_index[0]
#     # dst = data.edge_label_index[1]

#     # x_src = x[src]
#     # x_dst = x[dst]

#     # pair = torch.cat([x_src, x_dst, x_src * x_dst], dim = 1)

#     # return self.classifier(pair)

# TRAINING

In [ ]:
from tqdm import tqdm

def train_epoch(model, loader, optimizer, device):
  model.train()
  total_loss, total_edges = 0, 0

  # counts_tensor = torch.tensor([counts[i] for i in range(num_relations)], dtype = torch.float)
  # weights = 1/counts_tensor
  # weights = (weights / weights.sum()).to(device)

  for data in tqdm(loader, desc = "Training", leave=False):
    data = data.to(device)
    optimizer.zero_grad()

    logits = model(data)

    # loss = F.cross_entropy(logits, data.edge_label, weight=weights)
    loss = F.cross_entropy(logits, data.edge_label)

    loss.backward()
    optimizer.step()

    n = data.edge_label.size(0)

    total_loss += loss.item() * n

    total_edges += n

  return total_loss / total_edges

In [ ]:
def evaluate(model, loader, device):
  model.eval()
  total_loss, total_edges = 0, 0

  all_preds, all_labels = [], []

  with torch.no_grad():
    for data in loader:
      data = data.to(device)

      logits = model(data)

      loss = F.cross_entropy(logits, data.edge_label)

      n = data.edge_label.size(0)
      total_loss += loss.item() * n
      total_edges += n

      all_preds.append(logits.argmax(dim = -1).cpu())
      all_labels.append(data.edge_label.cpu())

  all_preds = torch.cat(all_preds).numpy()
  all_labels = torch.cat(all_labels).numpy()

  avg_loss = total_loss / total_edges
  accuracy = (all_preds == all_labels).mean()
  macro_f1 = f1_score(all_labels, all_preds, average = "macro")
  report = classification_report(all_labels, all_preds, target_names = label_names, zero_division = 0)

  return avg_loss, accuracy, macro_f1, report

In [ ]:
from sklearn.utils import shuffle
def run_training(families, directed, num_relations, in_dim, hidden_dim = 64, epochs = 150, lr = 1e-3, weight_decay = 1e-4, batch_size = 32, val_split = 0.15, test_split = 0.15, patience = 20, label_names = None, device = None, save_path = "family_relations_model.pt"):
  if device is None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  print(f"Using device: {device}")

  n = len(families)
  idx = torch.randperm(n).tolist()

  n_test = int(n * test_split)
  n_val = int(n * val_split)

  test_families = [families[i] for i in idx[:n_test]]
  val_families = [families[i] for i in idx[n_test:n_test + n_val]] if n_val > 0 else test_families
  train_families = [families[i] for i in idx[n_test + n_val:]]

  train_set = FamilyDataset(train_families, directed, num_relations)
  val_set = FamilyDataset(val_families, directed, num_relations)
  test_set = FamilyDataset(test_families, directed, num_relations)

  print(f"Families - train: {len(train_families)} | val: {len(val_families)} | test: {len(test_families)}")
  print(f"Samples - train: {len(train_set)}   | val: {len(val_set)}   | test: {len(test_set)}")

  # validate_dataset(train_set, num_relations, "train")
  # validate_dataset(val_set,   num_relations, "val")
  # validate_dataset(test_set,  num_relations, "test")

  train_loader = DataLoader(train_set, batch_size = batch_size, shuffle = True)
  val_loader = DataLoader(val_set, batch_size = batch_size)
  test_loader = DataLoader(test_set, batch_size = batch_size)

  model = FamilyRelationGNN(in_dim, hidden_dim, num_relations).to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr = lr, weight_decay = weight_decay)

  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience = patience // 3, factor = 0.5)

  best_val_f1, best_state, no_improve = -1, None, 0

  for epoch in range(1, epochs + 1):
    print(f"Epoch {epoch}/{epochs}")

    train_loss = train_epoch(model, train_loader, optimizer, device)

    val_loss, val_acc, val_f1, _ = evaluate(model, val_loader, device)

    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

    if val_f1 > best_val_f1:
      best_val_f1 = val_f1
      # best_state = copy.deepcopy(model.state_dict())
      best_state  = {k: v.clone() for k, v in model.state_dict().items()}
      no_improve = 0

    else:
      no_improve += 1

    if no_improve >= patience:
      print(f"Early stopping at epoch {epoch} - Best val F1: {best_val_f1:.4f}")
      break

  model.load_state_dict(best_state)
  test_loss, test_acc, test_f1, test_report = evaluate(model, test_loader, device)

  print(f"Test Loss: {test_loss:.4f} | Test F1: {test_f1:.4f} | Test Accuracy: {test_acc:.4f}")
  print(test_report)

  torch.save(model.state_dict(), save_path)
  print(f"Model saved to {save_path}")

  return model


In [ ]:
def find_asymmetric_pairs_from_files():
    asymmetric = set()
    family_ids = sorted([p.name for p in familyIDsDir.iterdir()])

    for id in tqdm(family_ids, desc="Scanning families"):
        mid_csv = familyIDsDir / id / "mid.csv"
        if not mid_csv.exists():
            continue

        try:
            df = pd.read_csv(mid_csv)
            mids = df["MID"].astype(int).tolist()

            row_map = {int(row["MID"]): row for _, row in df.iterrows()}

            for mid_a in mids:
                for mid_b in mids:
                    if mid_a == mid_b:
                        continue
                    if mid_a not in row_map or mid_b not in row_map:
                        continue

                    val_ab = row_map[mid_a].get(str(mid_b))
                    val_ba = row_map[mid_b].get(str(mid_a))

                    if not valid_relationship(val_ab) or not valid_relationship(val_ba):
                        continue

                    rel_ab = int(val_ab)
                    rel_ba = int(val_ba)

                    if rel_ab != rel_ba:
                        pair = tuple(sorted([rel_ab, rel_ba]))
                        asymmetric.add(pair)

        except Exception as e:
            print(f"\nFailed on {id}: {e}")

    return asymmetric

asymmetric_pairs = find_asymmetric_pairs_from_files()
print("Asymmetric relationship pairs (directed):")
for a, b in sorted(asymmetric_pairs):
    print(f"  {a} <-> {b}")


Scanning families: 100%|██████████| 980/980 [00:01<00:00, 634.67it/s]

Asymmetric relationship pairs (directed):
  1 <-> 4
  3 <-> 6


In [ ]:
directed = False
families = load_all_families(directed)
num_relations = max(r for f in families for _, _, r in f['edges']) + 1

Loading Families: 100%|██████████| 980/980 [00:02<00:00, 471.10it/s]


In [ ]:
print(f"num_relations: {num_relations}")

num_relations: 5


In [ ]:
from collections import Counter

all_labels = [r for f in families for _, _, r in f['edges']]
counts = Counter(all_labels)
for rel, count in sorted(counts.items()):
    print(f"  rel {rel}: {count}")


  rel 0: 4770
  rel 1: 2295
  rel 2: 9518
  rel 3: 2232


In [ ]:
directed = True
families = load_all_families(directed)
num_relations = max(r for f in families for _, _, r in f['edges']) + 1

from collections import Counter

all_labels = [r for f in families for _, _, r in f['edges']]
counts = Counter(all_labels)
for rel, count in sorted(counts.items()):
    print(f"  rel {rel}: {count}")


Loading Families: 100%|██████████| 980/980 [00:02<00:00, 471.98it/s]

  rel 0: 4760
  rel 1: 4770
  rel 2: 1147
  rel 3: 4758
  rel 4: 2232
  rel 5: 1148


In [ ]:

directed = True
num_relations = 6
# label_names = ["2 - sibling", "3 / 6", "1 / 4 - parent / child", "5 - spouse", "7 / 8"]
# num_relations = 8
# label_names = ['1', '2', '3', '4', '5', '6', '7', '8']
label_names = ['1', '2', '3', '4', '5', '6']

families = load_all_families(directed)
in_dim   = families[0]['x'].size(1)

model = run_training(
  families = families,
  directed = directed,
  num_relations = num_relations,
  in_dim = in_dim,
  label_names = label_names,
  # hidden_dim = 96,
  hidden_dim = 256,
  batch_size = 16,
  # val_split = 0,
  # test_split=0.1,
  # save_path = "family_relations_model_WEIGHTED_no_78.pt"
  save_path = "family_relations_model_no_78_NO_DIM_RED.pt"
)


Loading Families: 100%|██████████| 980/980 [00:02<00:00, 473.95it/s]


Using device: cuda
Families - train: 686 | val: 147 | test: 147
Samples - train: 3518   | val: 741   | test: 750
Epoch 1/150


Train Loss: 1.4790 | Val Loss: 1.2488 | Val F1: 0.4415 | Val Acc: 0.4627
Epoch 2/150


Train Loss: 1.1055 | Val Loss: 1.0859 | Val F1: 0.5297 | Val Acc: 0.5475
Epoch 3/150


Train Loss: 0.9095 | Val Loss: 1.0097 | Val F1: 0.5968 | Val Acc: 0.6014
Epoch 4/150


...

Train Loss: 0.1668 | Val Loss: 0.9616 | Val F1: 0.6812 | Val Acc: 0.7252
Epoch 28/150


Train Loss: 0.1583 | Val Loss: 0.9892 | Val F1: 0.6827 | Val Acc: 0.7246
Epoch 29/150


Train Loss: 0.1519 | Val Loss: 0.9860 | Val F1: 0.6816 | Val Acc: 0.7246
Epoch 30/150


Train Loss: 0.1504 | Val Loss: 0.9915 | Val F1: 0.6780 | Val Acc: 0.7232
Epoch 31/150


Train Loss: 0.1457 | Val Loss: 1.0215 | Val F1: 0.6694 | Val Acc: 0.7174
Early stopping at epoch 31 - Best val F1: 0.6867
Test Loss: 0.8868 | Test F1: 0.6633 | Test Accuracy: 0.7052
              precision    recall  f1-score   support

           1       0.76      0.76      0.76       695
           2       0.68      0.73      0.71       710
           3       0.74      0.47      0.57       164
           4       0.75      0.76      0.76       694
           5       0.56      0.61      0.59       338
           6       0.67      0.54      0.59       164

    accuracy                           0.71      2765
   macro avg       0.69      0.65      0.66      2765
weighted avg       0.71      0.71      0.70      2765

Model saved to family_relations_model_no_78_NO_DIM_RED.pt


In [ ]:

directed = False
num_relations = 5
label_names = ["2 - sibling", "3 / 6", "1 / 4 - parent / child", "5 - spouse", "7 / 8"]
# num_relations = 8
# label_names = ['0', '1', '2', '3', '4', '5', '6', '7']

families = load_all_families(directed)
in_dim   = families[0]['x'].size(1)

model = run_training(
  families = families,
  directed = directed,
  num_relations = num_relations,
  in_dim = in_dim,
  label_names = label_names,
)


Loading Families: 100%|██████████| 980/980 [00:02<00:00, 468.52it/s]


Using device: cuda
Families - train: 686 | val: 147 | test: 147
Samples - train: 3487   | val: 760   | test: 762
Epoch 1/150


Train Loss: 1.3200 | Val Loss: 1.2461 | Val F1: 0.1327
Epoch 2/150


Train Loss: 1.2342 | Val Loss: 1.2248 | Val F1: 0.1327
Epoch 3/150


Train Loss: 1.2006 | Val Loss: 1.1883 | Val F1: 0.1327
Epoch 4/150


...

Train Loss: 0.6058 | Val Loss: 0.9155 | Val F1: 0.5069
Epoch 85/150


Train Loss: 0.6067 | Val Loss: 0.9155 | Val F1: 0.5073
Epoch 86/150


Train Loss: 0.6062 | Val Loss: 0.9153 | Val F1: 0.5064
Epoch 87/150


Train Loss: 0.6071 | Val Loss: 0.9151 | Val F1: 0.5063
Epoch 88/150


Train Loss: 0.5975 | Val Loss: 0.9155 | Val F1: 0.5062
Early stopping at epoch 88 - Best val F1: 0.5104
Test Loss: 1.0825 | Test F1: 0.4548
                        precision    recall  f1-score   support

           2 - sibling       0.62      0.66      0.64       730
                 3 / 6       0.47      0.45      0.46       394
1 / 4 - parent / child       0.66      0.70      0.68      1456
            5 - spouse       0.50      0.48      0.49       358
                 7 / 8       0.00      0.00      0.00       100

              accuracy                           0.61      3038
             macro avg       0.45      0.46      0.45      3038
          weighted avg       0.59      0.61      0.60      3038

Model saved to family_relations_model.pt


In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
def validate_dataset(dataset, num_relations, name="dataset"):
    errors = []
    for i, data in enumerate(dataset):
        if data.edge_attr.numel() > 0:
            if data.edge_attr.min() < 0 or data.edge_attr.max() >= num_relations:
                errors.append(f"Sample {i}: edge_attr out of range — min={data.edge_attr.min()}, max={data.edge_attr.max()}")
        if data.edge_label.min() < 0 or data.edge_label.max() >= num_relations:
            errors.append(f"Sample {i}: edge_label out of range — min={data.edge_label.min()}, max={data.edge_label.max()}")

    if errors:
        print(f"\n{name} validation FAILED:")
        for e in errors:
            print(f"  {e}")
    else:
        print(f"{name}: all samples valid")

# validate_dataset(train_ds, num_relations, "train")
# validate_dataset(val_ds,   num_relations, "val")
# validate_dataset(test_ds,  num_relations, "test")

In [ ]:
class HeteroRelationshipGNN(nn.Module):
  def __init__(self, in_dim, hidden_dim, graph_labels, num_classes, dropout = 0.3, directed = True):
    super().__init__()

    self.graph_labels = graph_labels
    self.droupout = dropout

    self.input_proj = nn.linear(in_dim, hidden_dim)

    edge_types